## 00. Quick Start


In [1]:
print('Concept Portfolio V2 Lab — staged와 one-click은 동일 Core를 사용합니다.')
print('LIVE_TEST_LEVEL로 CORE / LEGAL_C1 / FULL_E2E / ONE_CLICK 중 하나만 선택하세요.')

Concept Portfolio V2 Lab — staged와 one-click은 동일 Core를 사용합니다.
LIVE_TEST_LEVEL로 CORE / LEGAL_C1 / FULL_E2E / ONE_CLICK 중 하나만 선택하세요.


## 01. Environment


In [2]:
import os, sys, json
from pathlib import Path
from IPython.display import display
SEARCH_ROOTS = [Path.cwd(), *Path.cwd().parents]
AI_ROOT = next((p for p in SEARCH_ROOTS if (p / 'app').is_dir()), None)
if AI_ROOT is None: AI_ROOT = next((p / 'ai' for p in SEARCH_ROOTS if (p / 'ai' / 'app').is_dir()))
if str(AI_ROOT) not in sys.path: sys.path.insert(0, str(AI_ROOT))
print({'python': sys.version.split()[0], 'aiRoot': str(AI_ROOT)})

{'python': '3.14.5', 'aiRoot': 'C:\\Users\\seewo\\Desktop\\big_proj_01\\new_3\\ai'}


## 02. MODE


In [3]:
MODE = 'LIVE'  # MOCK | REPLAY | LIVE
RECORDINGS_DIR = AI_ROOT / 'recordings' / 'concept_portfolio_v2'
print({'mode': MODE, 'liveExternalOperationsEnabled': MODE == 'LIVE'})

{'mode': 'LIVE', 'liveExternalOperationsEnabled': True}


## 03. Environment Check


In [4]:
LIVE_ENV_KEYS = ['AI_PROVIDER', 'AI_API_KEY', 'AI_MODEL', 'MOLEG_API_KEY', 'LEGAL_REGISTRY_VERSION']
env_status = {key: bool(os.getenv(key)) for key in LIVE_ENV_KEYS}
print(env_status if MODE == 'LIVE' else {'mode': MODE, 'message': '외부 환경변수 불필요'})

{'AI_PROVIDER': True, 'AI_API_KEY': True, 'AI_MODEL': True, 'MOLEG_API_KEY': True, 'LEGAL_REGISTRY_VERSION': True}


## 04. Schema Preflight


In [5]:
from app.concept_portfolio_v2 import ConceptPortfolioEngine, ProviderGateway, ProviderMode
from app.concept_portfolio_v2.adapters import CurrentLegalAdapter
from app.concept_portfolio_v2.diagnostics.notebook_view import *
gateway = ProviderGateway(MODE, recordings_dir=RECORDINGS_DIR)
engine = ConceptPortfolioEngine(MODE, gateway=gateway)
schema_preflight = engine.schema_preflight_report()
display(show_schema_preflight(schema_preflight))
assert schema_preflight.status == 'PASS' and schema_preflight.providerCalls == 0

,스키마,상태,실패,Provider 호출
0,PlanDraftPool,PASS,[],0
1,ConceptCandidateDraft,PASS,[],0
2,SemanticDistinctnessResult,PASS,[],0
3,SemanticFidelityResult,PASS,[],0
4,SemanticArchitectureBatch,PASS,[],0
5,SemanticHypothesisBatch,PASS,[],0
6,BusinessRoleSemanticBatch,PASS,[],0
7,LegalFactDependencySemanticBatch,PASS,[],0
8,LegalFactCompletionPatch,PASS,[],0


## 05. Input


In [6]:
SCENARIO_FILE = AI_ROOT / 'fixtures' / 'concept_portfolio_v2' / 'live_scenarios.json'
SCENARIOS = {item['scenarioId']: item for item in json.loads(SCENARIO_FILE.read_text(encoding='utf-8'))}
LIVE_SCENARIO = 'FOOD_PHYSICAL_COMMERCE'
LIVE_TEST_LEVEL = 'FULL_E2E'  # CORE | LEGAL_C1 | FULL_E2E | ONE_CLICK
RUN_STAGED_CORE = LIVE_TEST_LEVEL in {'CORE', 'LEGAL_C1', 'FULL_E2E'}
RUN_STAGED_LEGAL = LIVE_TEST_LEVEL in {'LEGAL_C1', 'FULL_E2E'}
RUN_STAGED_FULL = LIVE_TEST_LEVEL == 'FULL_E2E'
scenario = SCENARIOS[LIVE_SCENARIO]
TEST_INPUT = {key: scenario[key] for key in ('ideaOverview', 'problem', 'targetUsers')}
MAX_CONCEPTS = 5
display({'scenario': LIVE_SCENARIO, 'testLevel': LIVE_TEST_LEVEL, 'domain': scenario['domain'],
         'expectedStructuralFeatures': scenario['expectedStructuralFeatures']})

{'scenario': 'FOOD_PHYSICAL_COMMERCE',
 'testLevel': 'FULL_E2E',
 'domain': 'Food physical commerce',
 'expectedStructuralFeatures': ['물리적 이행', '직접 운영과 파트너 역할 구분']}

## 06. Idea Brief Derivation


In [7]:
seed = engine.seed_adapter.adapt(TEST_INPUT)
idea_context = None
if RUN_STAGED_CORE:
    engine._reset()
    idea_context = await engine.derive_idea_brief(seed)
print({'ideaCallComplete': bool(idea_context), 'interpretationPresent': bool(seed.interpretation),
       'stagedCore': RUN_STAGED_CORE})

{'ideaCallComplete': True, 'interpretationPresent': True, 'stagedCore': True}


## 07. Safety


In [8]:
display(idea_context.safetyReview.model_dump(mode='json') if idea_context else {'status': 'SKIPPED'})
assert idea_context is None or idea_context.safetyReview.passed

{'decision': 'ALLOW',
 'categories': [],
 'restrictions': [],
 'userFacingReason': '이 아이디어는 안전하며, 개인 맞춤형 식재료 제공과 관련된 문제를 해결하는 데 기여할 수 있습니다.'}

## 08. AI가 이해한 아이디어


In [9]:
display(show_idea_interpretation(idea_context) if idea_context else {'status': 'SKIPPED'})

,항목,AI 이해 결과
0,interpretedProblem,1~2인 가구에서 발생하는 음식물 쓰레기 문제를 해결하기 위한 서비스.
1,interpretedTargetUsers,요리할 시간이 적고 식재료 낭비를 줄이고자 하는 1~2인 가구.
2,usageContext,소량의 식재료를 필요로 하는 소비자에게 맞춤형 서비스를 제공하는 상황.
3,industryCategory,식품 서비스
4,researchScope,소량 식재료 및 레시피 제공 서비스에 대한 시장 조사.
5,conciseIdeaDefinition,개인 맞춤형 소량 식재료와 활용 레시피를 제공하는 서비스.
6,targetRegionInterpretation,특정 지역은 명시되지 않음.
7,relevantKnownCompetitorContext,경쟁자는 명시되지 않음.


## 09. Readiness / Summary / commitments


In [10]:
display(show_idea_readiness(idea_context) if idea_context else {'status': 'SKIPPED'})

{'readiness': {'status': 'READY_FOR_REVIEW',
  'score': 0,
  'missingFieldKeys': []},
 'readinessDiagnostic': 'READINESS_INCONSISTENT',
 'userFacingSummary': '이 서비스는 개인 맞춤형 소량 식재료를 제공하고 남은 재료를 활용한 레시피를 안내하여 1~2인 가구의 음식물 쓰레기 문제를 해결하는 것을 목표로 합니다.',
 'commitmentCandidates': [],
 'contradictions': [],
 'questions': []}

## 10. Seed Analysis


In [11]:
analysis = await engine.analyze_seed(seed) if RUN_STAGED_CORE else None
display(show_seed_analysis(analysis) if analysis else {'status': 'SKIPPED'})

,구분,값
0,탐색 폭,EXPLORE
1,다양성 수용량,5
2,설명,선택 입력 LOCK 0개로 11개 설계 차원이 열려 있습니다. diversityCa...


## 11. Generic Opportunity Kernel


In [12]:
display(analysis.opportunityKernel.model_dump(mode='json') if analysis else {'status': 'SKIPPED'})

{'problemCore': '1~2인 가구에서 발생하는 음식물 쓰레기 문제를 해결하기 위한 서비스.',
 'targetCore': '요리할 시간이 적고 식재료 낭비를 줄이고자 하는 1~2인 가구.',
 'useContexts': ['소량의 식재료를 필요로 하는 소비자에게 맞춤형 서비스를 제공하는 상황.'],
 'intentComponents': ['개인 맞춤형 소량 식재료와 활용 레시피를 제공하는 서비스.'],
 'mustPreserve': ['1~2인 가구에서 발생하는 음식물 쓰레기 문제를 해결하기 위한 서비스.',
  '요리할 시간이 적고 식재료 낭비를 줄이고자 하는 1~2인 가구.',
  '개인 맞춤형 소량 식재료와 활용 레시피를 제공하는 서비스.'],
 'maySpecialize': ['핵심 대상의 의미 있는 하위 세그먼트',
  '핵심 사용 맥락의 구체화',
  '가치 제안 또는 offer의 구체화'],
 'forbiddenDriftSummary': '핵심 문제와 대상이 모두 무관한 기회로 교체되면 범위를 벗어납니다.'}

## 12. Design Space


In [13]:
display(show_design_space(analysis) if analysis else {'status': 'SKIPPED'})

,분류,필드,값
0,SOURCE_LOCK,ideaOverview,개인 맞춤형 소량 식재료를 제공하고 남은 재료 활용 레시피를 안내하는 서비스
1,SOURCE_LOCK,problem,1~2인 가구가 큰 포장 단위 때문에 식재료를 남기고 음식물 쓰레기가 발생한다
2,SOURCE_LOCK,targetUsers,요리할 시간이 적고 낭비를 줄이고 싶은 1~2인 가구
3,SEMANTIC_ANCHOR,ideaOverview,개인 맞춤형 소량 식재료를 제공하고 남은 재료 활용 레시피를 안내하는 서비스
4,SEMANTIC_ANCHOR,problem,1~2인 가구가 큰 포장 단위 때문에 식재료를 남기고 음식물 쓰레기가 발생한다
5,SEMANTIC_ANCHOR,targetUsers,요리할 시간이 적고 낭비를 줄이고 싶은 1~2인 가구
6,OPEN,solutionMechanism,변경 가능
7,OPEN,valueDelivery,변경 가능
8,OPEN,operatingModel,변경 가능
9,OPEN,supplyStructure,변경 가능


## 13. Generate and Adaptively Replenish Plan Pool


In [14]:
plan_validation = (await engine.prepare_portfolio_plans(seed, analysis, max_concepts=MAX_CONCEPTS)
                   if RUN_STAGED_CORE else None)
plans = engine._last_plan_pool if plan_validation else []
print({'totalPlans': len(plans),
       'planningRounds': plan_validation.planningRounds if plan_validation else 0,
       'replenishmentRequested': plan_validation.replenishmentRequested if plan_validation else 0})

{'totalPlans': 6, 'planningRounds': 1, 'replenishmentRequested': 0}


## 14. Plan Count / Adaptive Replenishment Check


In [15]:
display(show_plan_pool_status(engine._last_plan_pool_status) if plan_validation else {'status': 'SKIPPED'})
display({'planningRounds': plan_validation.planningRounds if plan_validation else 0,
         'replenishmentRequested': plan_validation.replenishmentRequested if plan_validation else 0,
         'adaptiveReplenishmentUsed': bool(plan_validation and plan_validation.planningRounds > 1)})

,requestedPoolSize,returnedPoolSize,initialTarget,reserveTarget,reserveAvailable,status
0,7,6,5,2,1,RESERVE_SHORTFALL


{'planningRounds': 1,
 'replenishmentRequested': 0,
 'adaptiveReplenishmentUsed': False}

## 15. Korean Plan Display


In [16]:
display(show_portfolio_plans(plan_validation.acceptedPlans + plan_validation.reservePlans)
        if plan_validation else {'status': 'SKIPPED'})

,planId,제목,선택 상태,selectionScore,selectionReason,relationToPortfolio,Concept Family,Target Thesis,Use Context,Value Thesis,Offer Thesis,Solution Thesis,Architecture,비교 가치
0,P1,소량 맞춤형 식재료 서비스,SELECTED,0.8088,Opportunity fit과 Concept clarity가 가장 높은 대표안,PORTFOLIO_SEED,플랫폼 인프라 · 파트너 네트워크,요리할 시간이 적고 식재료 낭비를 줄이고자 하는 1~2인 가구,소량의 식재료를 필요로 하는 소비자에게 맞춤형 서비스를 제공하는 상황,필요한 만큼의 식재료를 제공하여 음식물 쓰레기를 줄이고 요리를 쉽게 할 수 있도록 ...,소량의 식재료와 활용 레시피를 제공하여 요리의 번거로움을 줄인다.,개인 맞춤형으로 필요한 식재료를 소량으로 제공하여 낭비를 최소화한다.,"{'businessRole': 'PLATFORM_INFRASTRUCTURE', 'o...",1~2인 가구의 식재료 낭비 문제를 해결하기 위한 혁신적인 접근 방식.
1,P4,1~2인 가구 전용 식재료 구독 서비스,SELECTED,0.7790,현재 Portfolio에 주요 사업 선택의 비교 범위를 추가,DISTINCT,플랫폼 인프라 · 파트너 네트워크,요리할 시간이 적고 식재료 낭비를 줄이고자 하는 1~2인 가구,정기적으로 필요한 식재료를 제공하는 상황,정기 구독을 통해 필요한 만큼의 식재료를 제공하여 낭비를 줄인다.,1~2인 가구에 최적화된 소량 식재료를 정기적으로 제공한다.,구독 모델을 통해 고객의 필요에 맞춘 식재료를 제공한다.,"{'businessRole': 'PLATFORM_INFRASTRUCTURE', 'o...",1~2인 가구의 식재료 낭비 문제를 해결하기 위한 전용 서비스.
2,P3,레시피 기반 식재료 추천 서비스,SELECTED,0.6885,현재 Portfolio에 주요 사업 선택의 비교 범위를 추가,DISTINCT,플랫폼 인프라 · 파트너 네트워크,요리할 시간이 적고 식재료 낭비를 줄이고자 하는 1~2인 가구,레시피에 따라 필요한 식재료를 추천하는 상황,레시피에 맞춘 소량의 식재료를 제공하여 요리의 효율성을 높인다.,고객이 선택한 레시피에 맞춰 필요한 재료를 소량으로 제공한다.,레시피 기반으로 식재료를 추천하여 낭비를 최소화한다.,"{'businessRole': 'PLATFORM_INFRASTRUCTURE', 'o...",레시피에 맞춘 식재료 제공으로 요리의 효율성을 높이기 위한 서비스.
3,P5,요리 시간 단축을 위한 식재료 서비스,SELECTED,0.5421,현재 Portfolio에 주요 사업 선택의 비교 범위를 추가,DISTINCT,플랫폼 인프라 · 파트너 네트워크,요리할 시간이 부족한 1~2인 가구,요리 시간을 단축하고 싶은 소비자에게 맞춤형 서비스를 제공하는 상황,소량의 식재료를 제공하여 요리 시간을 단축하고 낭비를 줄인다.,필요한 재료를 소량으로 제공하여 요리의 번거로움을 줄인다.,소량의 재료로 구성된 패키지를 제공하여 요리 시간을 단축한다.,"{'businessRole': 'PLATFORM_INFRASTRUCTURE', 'o...",요리 시간을 단축하고 식재료 낭비를 줄이기 위한 혁신적인 서비스.
4,P6,신선한 식재료 제공 서비스,SELECTED,0.4551,현재 Portfolio에 주요 사업 선택의 비교 범위를 추가,DISTINCT,플랫폼 인프라 · 파트너 네트워크,요리할 시간이 적고 식재료 낭비를 줄이고자 하는 1~2인 가구,신선한 식재료를 필요로 하는 소비자에게 맞춤형 서비스를 제공하는 상황,신선한 소량의 식재료를 제공하여 요리의 효율성을 높인다.,신선한 재료를 소량으로 제공하여 요리의 번거로움을 줄인다.,신선한 식재료를 정기적으로 제공하여 낭비를 최소화한다.,"{'businessRole': 'PLATFORM_INFRASTRUCTURE', 'o...",신선한 식재료 제공으로 요리의 효율성을 높이기 위한 서비스.
5,P2,간편 요리 키트 서비스,RESERVE,0.5908,Selected Portfolio 대비 marginal value가 낮아 reser...,"DISTINCT,VARIANT",플랫폼 인프라 · 파트너 네트워크,요리할 시간이 부족한 1~2인 가구,간편하게 요리를 하고 싶은 소비자에게 맞춤형 키트를 제공하는 상황,간편한 요리 키트를 통해 요리 시간을 단축하고 낭비를 줄인다.,필요한 재료와 레시피를 한 번에 제공하여 요리를 쉽게 만든다.,소량의 재료로 구성된 요리 키트를 제공하여 요리의 번거로움을 줄인다.,"{'businessRole': 'PLATFORM_INFRASTRUCTURE', 'o...",요리 시간을 단축하고 식재료 낭비를 줄이기 위한 혁신적인 서비스.


## 16. Plan Lock/Intent Validation


In [17]:
display({'accepted': [p.planId for p in plan_validation.acceptedPlans] if plan_validation else [],
         'rejected': [p.model_dump(mode='json') for p in plan_validation.rejectedPlans]
                     if plan_validation else []})

{'accepted': ['P1', 'P4', 'P3', 'P5', 'P6'], 'rejected': []}

## 17. Portfolio Family / Variant / Distinct


In [18]:
display(show_plan_diversity(plan_validation.diversity) if plan_validation else {'status': 'SKIPPED'})

,A,B,판정,Family A,Family B,겹침,실질 차이,단계,semantic judge,관계 설명
0,P1,P2,DISTINCT,PLATFORM_INFRASTRUCTURE:PARTNER_NETWORK,PLATFORM_INFRASTRUCTURE:PARTNER_NETWORK,"businessRole, operatingModel, partnerModel, de...","targetSegmentThesis, useCaseThesis, valuePropo...",PRIMARY_BUSINESS_CHOICE,False,핵심 solution 또는 주요 Business Architecture 선택이 다릅니다.
1,P1,P3,DISTINCT,PLATFORM_INFRASTRUCTURE:PARTNER_NETWORK,PLATFORM_INFRASTRUCTURE:PARTNER_NETWORK,"businessRole, operatingModel, partnerModel, de...","useCaseThesis, valuePropositionThesis, offerTh...",PRIMARY_BUSINESS_CHOICE,False,핵심 solution 또는 주요 Business Architecture 선택이 다릅니다.
2,P2,P3,DISTINCT,PLATFORM_INFRASTRUCTURE:PARTNER_NETWORK,PLATFORM_INFRASTRUCTURE:PARTNER_NETWORK,"businessRole, operatingModel, partnerModel, de...","targetSegmentThesis, useCaseThesis, valuePropo...",PRIMARY_BUSINESS_CHOICE,False,핵심 solution 또는 주요 Business Architecture 선택이 다릅니다.
3,P1,P4,DISTINCT,PLATFORM_INFRASTRUCTURE:PARTNER_NETWORK,PLATFORM_INFRASTRUCTURE:PARTNER_NETWORK,"businessRole, operatingModel, partnerModel, de...","useCaseThesis, valuePropositionThesis, offerTh...",PRIMARY_BUSINESS_CHOICE,False,핵심 solution 또는 주요 Business Architecture 선택이 다릅니다.
4,P2,P4,DISTINCT,PLATFORM_INFRASTRUCTURE:PARTNER_NETWORK,PLATFORM_INFRASTRUCTURE:PARTNER_NETWORK,"businessRole, operatingModel, partnerModel, de...","targetSegmentThesis, useCaseThesis, valuePropo...",PRIMARY_BUSINESS_CHOICE,False,핵심 solution 또는 주요 Business Architecture 선택이 다릅니다.
5,P3,P4,DISTINCT,PLATFORM_INFRASTRUCTURE:PARTNER_NETWORK,PLATFORM_INFRASTRUCTURE:PARTNER_NETWORK,"businessRole, operatingModel, partnerModel, de...","useCaseThesis, valuePropositionThesis, offerTh...",PRIMARY_BUSINESS_CHOICE,False,핵심 solution 또는 주요 Business Architecture 선택이 다릅니다.
6,P1,P5,DISTINCT,PLATFORM_INFRASTRUCTURE:PARTNER_NETWORK,PLATFORM_INFRASTRUCTURE:PARTNER_NETWORK,"businessRole, operatingModel, partnerModel, de...","targetSegmentThesis, useCaseThesis, valuePropo...",PRIMARY_BUSINESS_CHOICE,False,핵심 solution 또는 주요 Business Architecture 선택이 다릅니다.
7,P2,P5,VARIANT,PLATFORM_INFRASTRUCTURE:PARTNER_NETWORK,PLATFORM_INFRASTRUCTURE:PARTNER_NETWORK,"businessRole, operatingModel, partnerModel, de...","useCaseThesis, valuePropositionThesis, offerTh...",MEANINGFUL_THESIS_VARIANT,False,Architecture family는 유사하지만 target/use case/val...
8,P3,P5,DISTINCT,PLATFORM_INFRASTRUCTURE:PARTNER_NETWORK,PLATFORM_INFRASTRUCTURE:PARTNER_NETWORK,"businessRole, operatingModel, partnerModel, de...","targetSegmentThesis, useCaseThesis, valuePropo...",PRIMARY_BUSINESS_CHOICE,False,핵심 solution 또는 주요 Business Architecture 선택이 다릅니다.
9,P4,P5,DISTINCT,PLATFORM_INFRASTRUCTURE:PARTNER_NETWORK,PLATFORM_INFRASTRUCTURE:PARTNER_NETWORK,"businessRole, operatingModel, partnerModel, de...","targetSegmentThesis, useCaseThesis, valuePropo...",PRIMARY_BUSINESS_CHOICE,False,핵심 solution 또는 주요 Business Architecture 선택이 다릅니다.


## 18. Selected + Reserve Plans


In [19]:
selected_plans = plan_validation.acceptedPlans if plan_validation else []
reserve_plans = plan_validation.reservePlans if plan_validation else []
display(show_portfolio_plans(selected_plans + reserve_plans))
display({'selected': [p.planId for p in selected_plans], 'reserve': [p.planId for p in reserve_plans]})

,planId,제목,선택 상태,selectionScore,selectionReason,relationToPortfolio,Concept Family,Target Thesis,Use Context,Value Thesis,Offer Thesis,Solution Thesis,Architecture,비교 가치
0,P1,소량 맞춤형 식재료 서비스,SELECTED,0.8088,Opportunity fit과 Concept clarity가 가장 높은 대표안,PORTFOLIO_SEED,플랫폼 인프라 · 파트너 네트워크,요리할 시간이 적고 식재료 낭비를 줄이고자 하는 1~2인 가구,소량의 식재료를 필요로 하는 소비자에게 맞춤형 서비스를 제공하는 상황,필요한 만큼의 식재료를 제공하여 음식물 쓰레기를 줄이고 요리를 쉽게 할 수 있도록 ...,소량의 식재료와 활용 레시피를 제공하여 요리의 번거로움을 줄인다.,개인 맞춤형으로 필요한 식재료를 소량으로 제공하여 낭비를 최소화한다.,"{'businessRole': 'PLATFORM_INFRASTRUCTURE', 'o...",1~2인 가구의 식재료 낭비 문제를 해결하기 위한 혁신적인 접근 방식.
1,P4,1~2인 가구 전용 식재료 구독 서비스,SELECTED,0.7790,현재 Portfolio에 주요 사업 선택의 비교 범위를 추가,DISTINCT,플랫폼 인프라 · 파트너 네트워크,요리할 시간이 적고 식재료 낭비를 줄이고자 하는 1~2인 가구,정기적으로 필요한 식재료를 제공하는 상황,정기 구독을 통해 필요한 만큼의 식재료를 제공하여 낭비를 줄인다.,1~2인 가구에 최적화된 소량 식재료를 정기적으로 제공한다.,구독 모델을 통해 고객의 필요에 맞춘 식재료를 제공한다.,"{'businessRole': 'PLATFORM_INFRASTRUCTURE', 'o...",1~2인 가구의 식재료 낭비 문제를 해결하기 위한 전용 서비스.
2,P3,레시피 기반 식재료 추천 서비스,SELECTED,0.6885,현재 Portfolio에 주요 사업 선택의 비교 범위를 추가,DISTINCT,플랫폼 인프라 · 파트너 네트워크,요리할 시간이 적고 식재료 낭비를 줄이고자 하는 1~2인 가구,레시피에 따라 필요한 식재료를 추천하는 상황,레시피에 맞춘 소량의 식재료를 제공하여 요리의 효율성을 높인다.,고객이 선택한 레시피에 맞춰 필요한 재료를 소량으로 제공한다.,레시피 기반으로 식재료를 추천하여 낭비를 최소화한다.,"{'businessRole': 'PLATFORM_INFRASTRUCTURE', 'o...",레시피에 맞춘 식재료 제공으로 요리의 효율성을 높이기 위한 서비스.
3,P5,요리 시간 단축을 위한 식재료 서비스,SELECTED,0.5421,현재 Portfolio에 주요 사업 선택의 비교 범위를 추가,DISTINCT,플랫폼 인프라 · 파트너 네트워크,요리할 시간이 부족한 1~2인 가구,요리 시간을 단축하고 싶은 소비자에게 맞춤형 서비스를 제공하는 상황,소량의 식재료를 제공하여 요리 시간을 단축하고 낭비를 줄인다.,필요한 재료를 소량으로 제공하여 요리의 번거로움을 줄인다.,소량의 재료로 구성된 패키지를 제공하여 요리 시간을 단축한다.,"{'businessRole': 'PLATFORM_INFRASTRUCTURE', 'o...",요리 시간을 단축하고 식재료 낭비를 줄이기 위한 혁신적인 서비스.
4,P6,신선한 식재료 제공 서비스,SELECTED,0.4551,현재 Portfolio에 주요 사업 선택의 비교 범위를 추가,DISTINCT,플랫폼 인프라 · 파트너 네트워크,요리할 시간이 적고 식재료 낭비를 줄이고자 하는 1~2인 가구,신선한 식재료를 필요로 하는 소비자에게 맞춤형 서비스를 제공하는 상황,신선한 소량의 식재료를 제공하여 요리의 효율성을 높인다.,신선한 재료를 소량으로 제공하여 요리의 번거로움을 줄인다.,신선한 식재료를 정기적으로 제공하여 낭비를 최소화한다.,"{'businessRole': 'PLATFORM_INFRASTRUCTURE', 'o...",신선한 식재료 제공으로 요리의 효율성을 높이기 위한 서비스.
5,P2,간편 요리 키트 서비스,RESERVE,0.5908,Selected Portfolio 대비 marginal value가 낮아 reser...,"DISTINCT,VARIANT",플랫폼 인프라 · 파트너 네트워크,요리할 시간이 부족한 1~2인 가구,간편하게 요리를 하고 싶은 소비자에게 맞춤형 키트를 제공하는 상황,간편한 요리 키트를 통해 요리 시간을 단축하고 낭비를 줄인다.,필요한 재료와 레시피를 한 번에 제공하여 요리를 쉽게 만든다.,소량의 재료로 구성된 요리 키트를 제공하여 요리의 번거로움을 줄인다.,"{'businessRole': 'PLATFORM_INFRASTRUCTURE', 'o...",요리 시간을 단축하고 식재료 낭비를 줄이기 위한 혁신적인 서비스.


{'selected': ['P1', 'P4', 'P3', 'P5', 'P6'], 'reserve': ['P2']}

## 19. Candidate 1


In [20]:
candidate_one = (await engine.expand_plan(seed, selected_plans[0], 1)
                 if RUN_STAGED_CORE and selected_plans else None)
display(show_candidates([candidate_one]) if candidate_one else [])

,candidateId,lineageId,parentCandidateId,이름,핵심 작동방식,family,descriptor,수익,운영
0,C1,L1,None,소량 맞춤형 식재료 서비스,사용자가 원하는 요리와 필요한 재료를 입력하면 맞춤형 식재료를 제공한다.,기타 역할 · 기타 운영,{'thesis': {'targetSegmentThesis': '요리할 시간이 적고...,구독 모델을 통한 정기 배송,"온라인 플랫폼을 통해 주문을 받고, 물류 시스템을 통해 배송한다."


## 20. Candidate 1 Korean/Governance


In [21]:
candidate_one_reports = []
display({'candidateId': candidate_one.candidateId if candidate_one else None,
         'status': 'PENDING_FULL_CANDIDATE_RECOVERY'})

{'candidateId': 'C1', 'status': 'PENDING_FULL_CANDIDATE_RECOVERY'}

## 21. Candidate 1 Actual Generic Descriptor


In [22]:
display(show_concept_descriptors([candidate_one]) if candidate_one else [])

,entityId,family,dimension,code,confidence,source
0,C1,OTHER:OTHER,businessRole,OTHER,LOW,UNKNOWN
1,C1,OTHER:OTHER,operatingModel,OTHER,LOW,UNKNOWN
2,C1,OTHER:OTHER,partnerModel,PARTNER_NETWORK,HIGH,RULE
3,C1,OTHER:OTHER,deliveryModel,PHYSICAL_DELIVERY,HIGH,RULE
4,C1,OTHER:OTHER,transactionModel,OTHER,LOW,UNKNOWN
5,C1,OTHER:OTHER,monetizationModel,OTHER,LOW,UNKNOWN
6,C1,OTHER:OTHER,customerInteractionModel,OTHER,LOW,UNKNOWN
7,C1,OTHER:OTHER,dataDependency,NONE,NaN,NaN
8,C1,OTHER:OTHER,physicalDependency,MATERIAL,NaN,NaN


## 22. Candidate 1 Fidelity


In [23]:
display({'candidateId': candidate_one.candidateId if candidate_one else None,
         'fidelity': '전체 Candidate Recovery 단계에서 semantic fallback 포함 검증'})

{'candidateId': 'C1',
 'fidelity': '전체 Candidate Recovery 단계에서 semantic fallback 포함 검증'}

## 23. Remaining Candidates


In [24]:
remaining_candidates = []
if RUN_STAGED_CORE:
    for i, plan in enumerate(selected_plans[1:], 2):
        remaining_candidates.append(await engine.expand_plan(seed, plan, i))
candidate_drafts = ([candidate_one] if candidate_one else []) + remaining_candidates
display(show_candidates(candidate_drafts))

,candidateId,lineageId,parentCandidateId,이름,핵심 작동방식,family,descriptor,수익,운영
0,C1,L1,None,소량 맞춤형 식재료 서비스,사용자가 원하는 요리와 필요한 재료를 입력하면 맞춤형 식재료를 제공한다.,기타 역할 · 기타 운영,{'thesis': {'targetSegmentThesis': '요리할 시간이 적고...,구독 모델을 통한 정기 배송,"온라인 플랫폼을 통해 주문을 받고, 물류 시스템을 통해 배송한다."
1,C2,L2,None,1~2인 가구 전용 식재료 구독 서비스,구독 모델을 통해 고객의 필요에 맞춘 식재료를 제공한다.,기타 역할 · 기타 운영,{'thesis': {'targetSegmentThesis': '요리할 시간이 적고...,구독 기반 수익 모델,고객의 요리 선호에 따라 맞춤형 식재료를 정기적으로 배송하는 모델.
2,C3,L3,None,레시피 기반 식재료 추천 서비스,고객이 선택한 레시피에 따라 필요한 재료를 자동으로 추천하고 소량으로 제공하여 낭비...,기타 역할 · 파트너 네트워크,{'thesis': {'targetSegmentThesis': '요리할 시간이 적고...,구독 모델을 통한 정기 배송,"온라인 플랫폼을 통해 주문을 받고, 물류 시스템을 통해 배송한다."
3,C4,L4,None,요리 시간 단축을 위한 식재료 서비스,소량의 재료로 구성된 패키지를 제공하여 요리 시간을 단축한다.,기타 역할 · 기타 운영,{'thesis': {'targetSegmentThesis': '요리할 시간이 적고...,구독 모델을 통한 정기 배송,주문 후 1~2일 이내에 배송하여 신선도를 유지한다.
4,C5,L5,None,신선한 식재료 제공 서비스,신선한 식재료를 정기적으로 제공하여 낭비를 최소화한다.,기타 역할 · 기타 운영,{'thesis': {'targetSegmentThesis': '요리할 시간이 적고...,구독 모델을 통한 정기 수익,"온라인 플랫폼을 통해 주문을 받고, 물류 시스템을 통해 배송한다."


## 24. Candidate Actual Generic Descriptors


In [25]:
display(show_concept_descriptors(candidate_drafts))

,entityId,family,dimension,code,confidence,source
0,C1,OTHER:OTHER,businessRole,OTHER,LOW,UNKNOWN
1,C1,OTHER:OTHER,operatingModel,OTHER,LOW,UNKNOWN
2,C1,OTHER:OTHER,partnerModel,PARTNER_NETWORK,HIGH,RULE
3,C1,OTHER:OTHER,deliveryModel,PHYSICAL_DELIVERY,HIGH,RULE
4,C1,OTHER:OTHER,transactionModel,OTHER,LOW,UNKNOWN
5,C1,OTHER:OTHER,monetizationModel,OTHER,LOW,UNKNOWN
6,C1,OTHER:OTHER,customerInteractionModel,OTHER,LOW,UNKNOWN
7,C1,OTHER:OTHER,dataDependency,NONE,NaN,NaN
8,C1,OTHER:OTHER,physicalDependency,MATERIAL,NaN,NaN
9,C2,OTHER:OTHER,businessRole,OTHER,LOW,UNKNOWN


## 25. Candidate Recovery / Portfolio Relations


In [26]:
candidate_preparation = (await engine.prepare_candidate_portfolio(
    seed, plan_validation, max_concepts=MAX_CONCEPTS, initial_candidates=candidate_drafts)
    if RUN_STAGED_CORE and plan_validation else None)
candidates = candidate_preparation.candidates if candidate_preparation else []
candidate_reports = candidate_preparation.reports if candidate_preparation else []
display(show_candidate_recovery(candidate_preparation) if candidate_preparation else {'status': 'SKIPPED'})
candidate_pairwise = [engine.compare_candidates(candidates[i], candidates[j])
                      for i in range(len(candidates)) for j in range(i + 1, len(candidates))]
display(show_plan_diversity(candidate_pairwise))

{'summary':    candidateGenerated  candidateAcceptedInitially  candidateRegenerated  \
 0                   5                           5                     0   
 
    candidateRecovered  reservePlansActivated  candidateRecoveryReplans  \
 0                   0                      0                         0   
 
    finalCandidatePortfolio  
 0                        5  ,
 'attempts':   candidateId  schemaValid  hardLockPreserved  semanticAnchorPreserved  \
 0          C1         True               True                     True   
 1          C2         True               True                     True   
 2          C3         True               True                     True   
 3          C4         True               True                     True   
 4          C5         True               True                     True   
 
    planFidelity anchorDecision fidelityDecision  contentLanguageValid  \
 0          True           PASS             PASS                  True   
 1        

,A,B,판정,Family A,Family B,겹침,실질 차이,단계,semantic judge,관계 설명
0,C1,C2,DISTINCT,PLATFORM_INFRASTRUCTURE:PARTNER_NETWORK,PLATFORM_INFRASTRUCTURE:PARTNER_NETWORK,"businessRole, operatingModel, partnerModel, de...","valuePropositionThesis, offerThesis, solutionT...",PRIMARY_BUSINESS_CHOICE,False,핵심 solution 또는 주요 Business Architecture 선택이 다릅니다.
1,C1,C3,DISTINCT,PLATFORM_INFRASTRUCTURE:PARTNER_NETWORK,PLATFORM_INFRASTRUCTURE:PARTNER_NETWORK,"businessRole, operatingModel, partnerModel, de...","valuePropositionThesis, offerThesis, solutionT...",PRIMARY_BUSINESS_CHOICE,False,핵심 solution 또는 주요 Business Architecture 선택이 다릅니다.
2,C1,C4,DISTINCT,PLATFORM_INFRASTRUCTURE:PARTNER_NETWORK,PLATFORM_INFRASTRUCTURE:PARTNER_NETWORK,"businessRole, operatingModel, partnerModel, de...","valuePropositionThesis, offerThesis, solutionT...",PRIMARY_BUSINESS_CHOICE,False,핵심 solution 또는 주요 Business Architecture 선택이 다릅니다.
3,C1,C5,DISTINCT,PLATFORM_INFRASTRUCTURE:PARTNER_NETWORK,PLATFORM_INFRASTRUCTURE:PARTNER_NETWORK,"businessRole, operatingModel, partnerModel, de...","valuePropositionThesis, offerThesis, solutionT...",PRIMARY_BUSINESS_CHOICE,False,핵심 solution 또는 주요 Business Architecture 선택이 다릅니다.
4,C2,C3,DISTINCT,PLATFORM_INFRASTRUCTURE:PARTNER_NETWORK,PLATFORM_INFRASTRUCTURE:PARTNER_NETWORK,"businessRole, operatingModel, partnerModel, de...","valuePropositionThesis, solutionThesis",PRIMARY_BUSINESS_CHOICE,False,핵심 solution 또는 주요 Business Architecture 선택이 다릅니다.
5,C2,C4,DISTINCT,PLATFORM_INFRASTRUCTURE:PARTNER_NETWORK,PLATFORM_INFRASTRUCTURE:PARTNER_NETWORK,"businessRole, operatingModel, partnerModel, de...","valuePropositionThesis, solutionThesis",PRIMARY_BUSINESS_CHOICE,False,핵심 solution 또는 주요 Business Architecture 선택이 다릅니다.
6,C2,C5,DISTINCT,PLATFORM_INFRASTRUCTURE:PARTNER_NETWORK,PLATFORM_INFRASTRUCTURE:PARTNER_NETWORK,"businessRole, operatingModel, partnerModel, de...","valuePropositionThesis, solutionThesis",PRIMARY_BUSINESS_CHOICE,False,핵심 solution 또는 주요 Business Architecture 선택이 다릅니다.
7,C3,C4,DISTINCT,PLATFORM_INFRASTRUCTURE:PARTNER_NETWORK,PLATFORM_INFRASTRUCTURE:PARTNER_NETWORK,"businessRole, operatingModel, partnerModel, de...","valuePropositionThesis, offerThesis, solutionT...",PRIMARY_BUSINESS_CHOICE,False,핵심 solution 또는 주요 Business Architecture 선택이 다릅니다.
8,C3,C5,DISTINCT,PLATFORM_INFRASTRUCTURE:PARTNER_NETWORK,PLATFORM_INFRASTRUCTURE:PARTNER_NETWORK,"businessRole, operatingModel, partnerModel, de...",solutionThesis,PRIMARY_BUSINESS_CHOICE,False,핵심 solution 또는 주요 Business Architecture 선택이 다릅니다.
9,C4,C5,DISTINCT,PLATFORM_INFRASTRUCTURE:PARTNER_NETWORK,PLATFORM_INFRASTRUCTURE:PARTNER_NETWORK,"businessRole, operatingModel, partnerModel, de...","valuePropositionThesis, solutionThesis",PRIMARY_BUSINESS_CHOICE,False,핵심 solution 또는 주요 Business Architecture 선택이 다릅니다.


## 26. Legal Fact Completeness + Business Design Completion + C1 Fact Pattern


In [27]:
prechecks = [engine.legal_precheck(item) for item in candidates] if RUN_STAGED_LEGAL else []
display(show_legal_precheck(prechecks))
legal_preparation = (await engine.prepare_legal_candidates(seed, candidates)
                     if RUN_STAGED_LEGAL and candidates else None)
candidates_before_legal = candidates
candidates = legal_preparation.candidates if legal_preparation else candidates
display({'factCompleteness': [item.model_dump(mode='json') for item in legal_preparation.reports]
                              if legal_preparation else [],
         'roleSemanticBatchCalls': legal_preparation.roleSemanticBatchCalls if legal_preparation else 0,
         'dependencySemanticBatchCalls': legal_preparation.dependencySemanticBatchCalls if legal_preparation else 0,
         'completionAttempted': legal_preparation.completionAttempted if legal_preparation else 0,
         'completionValidated': legal_preparation.completionValidated if legal_preparation else 0,
         'completionAccepted': legal_preparation.completionAccepted if legal_preparation else 0,
         'completionExhausted': legal_preparation.completionExhausted if legal_preparation else 0,
         'completionCompliance': [item.model_dump(mode='json') for item in legal_preparation.completionCompliance] if legal_preparation else [],
         'preLegalExclusions': legal_preparation.excludedCandidates if legal_preparation else []})
display(show_legal_fact_pattern(candidates[0].candidate, seed) if candidates else [])

,candidateId,label,directSeller,intermediary,regulatedPhysicalActivity,personalDataDependency,qualificationDependency,riskHints
0,C1,Structural risk precheck — not final legal review,False,False,True,False,False,[물리 활동]
1,C2,Structural risk precheck — not final legal review,False,False,True,False,False,[물리 활동]
2,C3,Structural risk precheck — not final legal review,False,False,True,False,False,[물리 활동]
3,C4,Structural risk precheck — not final legal review,False,False,True,False,False,[물리 활동]
4,C5,Structural risk precheck — not final legal review,True,False,True,False,False,[물리 활동]


ProviderFailure: LEGAL_FACT_DEPENDENCY_BATCH_IDENTITY_MISMATCH

## 27. Prepared Legal C1 Evidence Summary


In [ ]:
legal_adapter = CurrentLegalAdapter() if RUN_STAGED_LEGAL else None
legal_c1_input = (legal_adapter.task_input(candidates[0].candidate, seed)
                  if legal_adapter and candidates else None)
display({'candidateId': candidates[0].candidateId if candidates else None,
         'externalFacts': legal_c1_input['externalFactContext']['facts'] if legal_c1_input else [],
         'note': '공식 근거 수와 allowed index는 Full Legal 응답/실패 diagnostics에서 확인'})

## 28. Full Evidence Judgment — C1 Staged Smoke


In [ ]:
RUN_FULL_LEGAL_C1 = RUN_STAGED_LEGAL
legal_one = None
if RUN_FULL_LEGAL_C1 and candidates:
    try:
        legal_one = await engine.review_legal_candidate(seed, candidates[0])
        display(show_legal_result([legal_one]))
    except Exception:
        display(show_legal_failure(candidates[0].candidateId, engine.gateway))
else:
    print('SKIPPED — RUN_FULL_LEGAL_C1=True로 명시해야 실행됩니다.')

## 29. C1 Route + Staged Redesign/Compliance/Second Legal


In [ ]:
c1_portfolio, c1_legal_all, c1_required_inputs, c1_redesigned, c1_replanned = ([], [], [], 0, 0)
if legal_one and candidates:
    c1_portfolio, c1_legal_all, c1_required_inputs, c1_redesigned, c1_replanned = await engine.resolve_legal(
        seed, candidate_preparation.usedPlans, candidates[:1], [legal_one])
display({'initialRoute': legal_one.route.value if legal_one else 'SKIPPED',
         'redesignRequirements': legal_one.redesignRequirements if legal_one else [],
         'recoveryReviews': [item.model_dump(mode='json') for item in c1_legal_all[1:]],
         'requiredInputs': c1_required_inputs, 'terminalCandidates': len(c1_portfolio),
         'redesigned': c1_redesigned, 'replanned': c1_replanned})

## 30. Remaining 4 Legal + Exhaustive Recovery Summary


In [ ]:
RUN_REMAINING_LEGAL = RUN_STAGED_FULL
legal_remaining = []
portfolio, legal_all, required_inputs = (list(c1_portfolio), list(c1_legal_all), list(c1_required_inputs))
redesigned_count, replanned_count = c1_redesigned, c1_replanned
c1_terminal = bool(c1_portfolio or c1_required_inputs or (c1_legal_all and c1_legal_all[-1].route.value == 'SYSTEM_FAILURE'))
if RUN_REMAINING_LEGAL and c1_terminal and len(candidates) > 1:
    legal_remaining = await engine.review_legal(seed, candidates[1:])
    rest_portfolio, rest_legal, rest_inputs, rest_redesigned, rest_replanned = await engine.resolve_legal(
        seed, candidate_preparation.usedPlans, candidates[1:], legal_remaining)
    portfolio += rest_portfolio; legal_all += rest_legal; required_inputs += rest_inputs
    redesigned_count += rest_redesigned; replanned_count += rest_replanned
else:
    print('SKIPPED — C1이 정상 terminal에 도달한 후 remaining Legal을 실행합니다.')
legal_initial = ([legal_one] if legal_one else []) + legal_remaining
display({'recoveryTrace': [item.model_dump(mode='json') for item in legal_all
                           if item.candidateId not in {x.candidateId for x in legal_initial}],
         'requiredInputs': required_inputs, 'metrics': engine._legal_metrics})
print({'Plan Selected': len(selected_plans),
       'Candidate Generated': candidate_preparation.candidateGenerated if candidate_preparation else 0,
       'Candidate Valid Initially': candidate_preparation.candidateAcceptedInitially if candidate_preparation else 0,
       'Candidate Regenerated': candidate_preparation.candidateRegenerated if candidate_preparation else 0,
       'Candidate Recovered': candidate_preparation.candidateRecovered if candidate_preparation else 0,
       'Fact Completion Attempted': legal_preparation.completionAttempted if legal_preparation else 0,
       'Fact Completion Validated': legal_preparation.completionValidated if legal_preparation else 0,
       'Fact Completion Accepted': legal_preparation.completionAccepted if legal_preparation else 0,
       'Dependency Semantic Calls': legal_preparation.dependencySemanticBatchCalls if legal_preparation else 0,
       'Completion Compliance PASS': sum(item.status == 'PASS' for item in legal_preparation.completionCompliance) if legal_preparation else 0,
       'Legal Ready': len(candidates) if legal_preparation else 0,
       'Legal Reviewed': len(legal_initial),
       'Legal Accepted': sum(item.route.value == 'ACCEPT' for item in legal_all),
       'Legal Redesigned': redesigned_count, 'Legal Replanned': replanned_count,
       'Final Portfolio': len(portfolio)})

## 31. Replan


In [ ]:
display(show_replan(type('PortfolioView', (), {'concepts': portfolio})()))
print({'replanned': replanned_count, 'reserveAvailable': len(reserve_plans)})

## 32. Final Portfolio


In [ ]:
legal_terminal_status = ('READY_FULL' if len(portfolio) == MAX_CONCEPTS else
    'READY_LIMITED' if portfolio else
    'LEGAL_RECOVERY_COMPLETE_NO_ACCEPTED_CANDIDATE' if legal_initial and len(legal_initial) == len(candidates)
    else 'LEGAL_PENDING')
display(show_final_portfolio(type('PortfolioView', (), {'concepts': portfolio})()) if portfolio else {'status': legal_terminal_status})

## 33. Unresolved Candidate Summary


In [ ]:
display(show_required_inputs(required_inputs) if required_inputs else {'unresolved': []})

## 34. Manual Concept Selection


In [ ]:
SELECTED_CANDIDATE_ID = portfolio[0].candidateId if portfolio else None  # 사용자가 수정
selected_concept = next((item for item in portfolio if item.candidateId == SELECTED_CANDIDATE_ID), None)
print({'selectedCandidateId': SELECTED_CANDIDATE_ID})

## 35. 7 Hypotheses


In [ ]:
hypotheses = (engine.build_or_load_current_hypothesis_contract(selected_concept)
              if RUN_STAGED_FULL and selected_concept else [])
hypotheses = await engine.resolve_hypothesis_semantics(hypotheses) if hypotheses else []
display(show_hypotheses(hypotheses))
display(show_hypothesis_readiness(hypotheses))

## 36. Confirm / Edit


In [ ]:
CONFIRM_ALL_PROPOSED = True
HYPOTHESIS_EDITS = {
    # 'PRICE': '월 17,900원',
}
confirmed_hypotheses = engine.confirm_hypotheses(
    hypotheses, HYPOTHESIS_EDITS, confirm_all_proposed=CONFIRM_ALL_PROPOSED) if hypotheses else []
display(show_hypotheses(confirmed_hypotheses))
hypothesis_readiness = show_hypothesis_readiness(confirmed_hypotheses)
display(hypothesis_readiness)

## 37. Actual Delta Legal


In [ ]:
RUN_DELTA_LEGAL = True
delta_legal_result = None
if RUN_STAGED_FULL and RUN_DELTA_LEGAL and selected_concept and any(h.deltaLegalRequired for h in confirmed_hypotheses):
    delta_legal_result = await engine.review_delta_legal(seed, selected_concept, confirmed_hypotheses)
    confirmed_hypotheses = engine.mark_delta_legal_reviewed(confirmed_hypotheses, delta_legal_result)
display(delta_legal_result.model_dump(mode='json') if delta_legal_result else {'status': 'NOT_REQUIRED_OR_SKIPPED'})

## 38. Market Seed


In [ ]:
handoff = None
if selected_concept and legal_all and hypothesis_readiness['Ready For Handoff']:
    handoff = engine.build_downstream_handoff(seed, selected_concept, confirmed_hypotheses, legal_all)
display(handoff.marketAnalysisSeedSnapshot if handoff else {
    'status': 'NOT_READY', 'reason': hypothesis_readiness.get('reason'),
    'unresolvedHypotheses': hypothesis_readiness.get('unresolvedHypotheses', [])})

## 39. Marketing Source


In [ ]:
display(handoff.marketingSourceSnapshot if handoff else {
    'status': 'NOT_READY', 'reason': hypothesis_readiness.get('reason'),
    'unresolvedHypotheses': hypothesis_readiness.get('unresolvedHypotheses', [])})

## 40. Contract Compatibility


In [ ]:
display(show_downstream_handoff(handoff) if handoff else {
    'contract': 'NOT_READY', 'reason': hypothesis_readiness.get('reason'),
    'unresolvedHypotheses': hypothesis_readiness.get('unresolvedHypotheses', [])})

## 41. Trace


In [ ]:
display(show_trace(engine.trace))

## 42. Provider/Legal Usage


In [ ]:
display(show_provider_usage(engine.gateway.usage))
print('상위 외부 작업 수는 내부 AI/MOLEG 네트워크 호출 수와 동일하다고 주장하지 않습니다.')

## 43. Replay Manifest


In [ ]:
display(show_replay_manifest(engine.gateway))

## 44. One-click MOCK


In [ ]:
mock_result = None
if LIVE_TEST_LEVEL == 'FULL_E2E' and MODE != 'LIVE':
    mock_result = await ConceptPortfolioEngine('MOCK').run_full(
        TEST_INPUT, max_concepts=MAX_CONCEPTS, auto_confirm_hypotheses=True)
display(show_run_summary(mock_result) if mock_result else {'status': 'SKIPPED'})
assert mock_result is None or (mock_result.handoff and mock_result.handoff.contractStatus == 'CONTRACT_PASS')

## 45. One-click REPLAY


In [ ]:
RUN_ONE_CLICK_REPLAY = False
replay_result = None
if RUN_ONE_CLICK_REPLAY:
    replay_gateway = ProviderGateway('REPLAY', recordings_dir=RECORDINGS_DIR)
    replay_result = await ConceptPortfolioEngine('REPLAY', gateway=replay_gateway).run_full(
        TEST_INPUT, max_concepts=MAX_CONCEPTS, auto_confirm_hypotheses=False)
display(show_run_summary(replay_result) if replay_result else {'status': 'SKIPPED'})

## 46. One-click LIVE


In [ ]:
RUN_ONE_CLICK_LIVE = True
live_result = None
if RUN_ONE_CLICK_LIVE and LIVE_TEST_LEVEL == 'ONE_CLICK':
    assert MODE == 'LIVE', 'MODE=LIVE를 먼저 명시하세요.'
    one_click_gateway = ProviderGateway('LIVE', recordings_dir=RECORDINGS_DIR)
    one_click_engine = ConceptPortfolioEngine('LIVE', gateway=one_click_gateway)
    live_result = await one_click_engine.run_full(TEST_INPUT, max_concepts=MAX_CONCEPTS,
                                                  auto_confirm_hypotheses=False)
display(show_run_summary(live_result) if live_result else {'status': 'SKIPPED'})
if live_result:
    display(show_live_validation_summary(LIVE_SCENARIO, live_result))
    display(show_required_inputs(live_result))
    display(show_pre_legal_exclusions(live_result))
    display(show_legal_resolutions(live_result))
    if live_result.runStatus.value == 'FAILED':
        display(show_run_failure(live_result))
        display(show_provider_failure(one_click_engine.gateway))
        display(show_provider_usage(live_result.providerUsage))
        display(show_trace(live_result.trace[-20:]))
        display({'unresolvedCandidates': live_result.unresolvedCandidates,
                 'lastSuccessfulStage': live_result.failureDiagnostics.lastSuccessfulStage if live_result.failureDiagnostics else None,
                 'firstFailedStage': live_result.failureDiagnostics.firstFailedStage if live_result.failureDiagnostics else None})